In [1]:
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt, os, sys
import networkx as nx, math, glob
import pickle, copy

import torch
from torch_geometric.data import Data
from torch_geometric.utils.convert import from_networkx, to_networkx

from torch_geometric.data import Data
from torch_geometric.utils import degree, coalesce

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

In [2]:
with open("motorable_ky_aadt_dual_graph_nbc2.pkl", "rb") as f:
    rd_dual_graph = pickle.load(f)

In [3]:
nodeslist = list(rd_dual_graph.nodes)
if all(isinstance(n, int) for n in nodeslist):
    sorted_nodes = sorted(nodeslist)
    is_contiguous = sorted_nodes == list(range(len(nodeslist)))
    print("Are node IDs contiguous from 0 to N-1?:", is_contiguous) #required by pytorch geometric
else:
    print("Not all node IDs are integers — mapping needed.") #required

Not all node IDs are integers — mapping needed.


In [4]:
original_nodes = sorted(list(rd_dual_graph.nodes))
uid_to_index = {uid: i for i, uid in enumerate(original_nodes)}
index_to_uid = {i: uid for uid, i in uid_to_index.items()}

nx_dgraph_indexed = nx.DiGraph()

for uid, attr in rd_dual_graph.nodes(data=True):
    nx_dgraph_indexed.add_node(uid_to_index[uid], **attr)

for u, v, edge_attrs in rd_dual_graph.edges(data=True):
        nx_dgraph_indexed.add_edge(uid_to_index[u], uid_to_index[v], **edge_attrs)

In [5]:
edge_list = list(nx_dgraph_indexed.edges())
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

In [6]:
attr_keys = list(next(iter(nx_dgraph_indexed.nodes(data=True)))[1].keys())
node_attr_lists = {key: [] for key in attr_keys}

for node_id in range(len(nx_dgraph_indexed.nodes)):
    for key in attr_keys:
        node_attr_lists[key].append(nx_dgraph_indexed.nodes[node_id].get(key, None))

In [7]:
pyg_data = Data(edge_index=edge_index)
for key, val_list in node_attr_lists.items():
        pyg_data[key] = val_list

In [8]:
def remove_isolated_nodes(data: Data):
    # Step 1: Identify isolated nodes
    src, dst = data.edge_index
    deg_src = degree(src, data.num_nodes)
    deg_dst = degree(dst, data.num_nodes)
    connected_mask = (deg_src + deg_dst) > 0  # keep only connected nodes

    # Step 2: Build mapping from old index → new index
    old_to_new = {}
    new_index = 0
    for old_idx, keep in enumerate(connected_mask):
        if keep:
            old_to_new[old_idx] = new_index
            new_index += 1

    # Step 3: Filter node attributes (stored as lists)
    new_data = Data()
    for key, val in data.items():
        if key == 'edge_index':
            continue  # handled below
        if isinstance(val, list) and len(val) == data.num_nodes:
            new_data[key] = [val[i] for i in range(data.num_nodes) if connected_mask[i]]
        else:
            new_data[key] = val  # keep as-is

    # Step 4: Filter edges and remap node indices
    new_edges = []
    for u, v in data.edge_index.t().tolist():
        if connected_mask[u] and connected_mask[v]:
            new_edges.append((old_to_new[u], old_to_new[v]))

    new_data.edge_index = torch.tensor(new_edges, dtype=torch.long).t().contiguous()

    return new_data

In [9]:
def get_isolated_nodes_directed(data):
    # Compute in-degree and out-degree
    row, col = data.edge_index
    out_deg = degree(row, num_nodes=data.num_nodes)
    in_deg = degree(col, num_nodes=data.num_nodes)
    
    # Find nodes with both in-degree and out-degree = 0
    isolated_nodes = torch.where((out_deg == 0) & (in_deg == 0))[0]
    
    return isolated_nodes

In [10]:
#Before: check no isolated nodes exist
isolated = get_isolated_nodes_directed(pyg_data)
print(len(isolated))
isolated

144


tensor([  2190,  12020,  14929,  15149,  16414,  17444,  20419,  35832,  36149,
         36163,  38514,  38577,  41531,  43658,  51347,  55527,  61347,  66829,
         75346,  81661,  81802,  81885,  83586,  86549,  96580,  99554, 112831,
        115966, 117203, 117857, 118821, 123612, 125421, 126138, 127473, 128745,
        129154, 130432, 130476, 135755, 137747, 138194, 141278, 144006, 144204,
        144362, 144444, 148944, 155288, 155345, 156172, 157663, 161570, 162012,
        162307, 166067, 176117, 189972, 190020, 197848, 204979, 208463, 210618,
        224338, 225899, 226082, 228922, 230326, 231329, 232870, 233617, 236321,
        236400, 236563, 237475, 239822, 240744, 241157, 241292, 241400, 241480,
        244656, 247966, 248008, 249320, 249733, 250937, 251940, 254790, 255952,
        261004, 261894, 274960, 275139, 291754, 292629, 294985, 299087, 300171,
        301802, 305583, 305584, 306136, 307517, 310311, 311494, 312848, 313982,
        321666, 323514, 328006, 328352, 

In [11]:
pyg_data = remove_isolated_nodes(pyg_data)
pyg_data.edge_index = coalesce(pyg_data.edge_index, num_nodes=pyg_data.num_nodes)

In [12]:
#After: confirm no isolated nodes exist
isolated = get_isolated_nodes_directed(pyg_data)
print(len(isolated))
isolated

0


tensor([], dtype=torch.int64)

In [13]:
if pyg_data.validate():
    print("pyg_data object is properly defined")
print(f'Num node features: {pyg_data.num_node_features}')
print(f'Num edge features: {pyg_data.num_edge_features}')
print(f'Edge atributes: {pyg_data.edge_attrs()}')
print(f'Num node types (should be one; all nodes represent roadway links): {pyg_data.num_node_types}')
print(f'Does Graph contains isolated nodes?: {pyg_data.has_isolated_nodes()}')
print(f'Does Graph contains self loops?: {pyg_data.has_self_loops()}')
print(f'Graph directed?: {pyg_data.is_directed()}')
print(f'Graph edges coalesced: {pyg_data.is_coalesced()}')

pyg_data object is properly defined
Num node features: 0
Num edge features: 0
Edge atributes: ['edge_index']
Num node types (should be one; all nodes represent roadway links): 1
Does Graph contains isolated nodes?: False
Does Graph contains self loops?: False
Graph directed?: True
Graph edges coalesced: True


In [14]:
datacols = ['CO_NAME', 'TYPE_OP','AADT', 'Truck_AADT', 'Length','Mid_Lat', 'Mid_Long', 
'pop5min', 'DPop5min', 'Worker5min', 'Resident5min', 'HH5min', 'PerCapitaIncome5min', 
    'MedianAge5min', 'Employed5min', 'MdHHIn5min', 'Transport_MarterialMoving5min', 'Businesses5min', 
    'Employees5min', 'AgMn5min', 'Mining5min', 'Utilities5min', 'Construction5min', 'Manufacturing5min', 
    'Transportation_Warehousing5min', 'RlvntBusi5min',
'AvgSpeed', 'StdSpeed', 'Pcnt5Speed', 'Pcnt20Speed', 
'Pcnt50Speed', 'Pcnt85Speed', 'RURAL_URBAN', 'F_SYSTEM', 'INTERCHANGES', 'AT_GRADE_OTHER', 
'AT_GRADE_SIGNALS', 'AT_GRADE_SIGNS', 'Truck_Perc', 'TYPE_TERRAIN', 'THROUGH_LANES',
'LANESCRD','LANESNC', 'LANE_WIDTH', 'MEDIAN_TYPE', 'MEDIAN_WIDTH', 'SHLD_WIDTH_R',
'SHLD_WIDTH_L', 'ACCESS_CONTROL', 'SPEED_LIMIT_LWA', 'bcn_rac','zBCn_PR_ac', 'AADP', 'TT_BC']

In [15]:
pyg_data['Type_Operation'] = [
    'one-way highway' if val == '1' else
    'two-way highway' if val == '2' else
    'divided by median highway' if val == 'D' else val
    for val in pyg_data['TYPE_OP']
]

terrain_mapping = {
    1: 'Flat',
    2: 'Rolling',
    3: 'Mountainous'
}
pyg_data['Terrain_Type'] = [terrain_mapping.get(val, val) for val in pyg_data['TYPE_TERRAIN']]

median_mapping = {
    1: 'Concrete Barrier',
    2: 'Guardrail Barrier',
    3: 'Other Positive Barrier',
    4: 'Raised Non Mountable',
    5: 'Raised Mountable',
    6: 'Flush',
    7: 'Depressed',
    8: 'No Median'
}

pyg_data['MedianType'] = [median_mapping.get(val, val) for val in pyg_data['MEDIAN_TYPE']]

access_mapping = {
    1: 'Full',
    2: 'Partial',
    3: 'By Permit'
}

pyg_data['AccessControl'] = [access_mapping.get(val, val) for val in pyg_data['ACCESS_CONTROL']]

In [16]:
continuous_features = ['AADP', 'THROUGH_LANES', 'LANE_WIDTH','SHLD_WIDTH_R', 'SPEED_LIMIT_LWA', 
                       'F_SYSTEM', 'MdHHIn5min', 'pop5min', 'DPop5min', 'HH5min', 'Businesses5min',
                        'Worker5min', 'RlvntBusi5min', 'Resident5min',
                       'Employed5min', 'AvgSpeed', 'Pcnt85Speed',
                       'TT_BC']

In [17]:
pyg_data.x = torch.stack([
    torch.tensor(pyg_data[col], dtype=torch.float) 
    for col in continuous_features
], dim=1)
print(pyg_data.x.size())

torch.Size([404070, 18])


In [18]:
pyg_data.y = torch.tensor(pyg_data['AADT'], dtype=torch.float).unsqueeze(1)
print(pyg_data.y.size())

torch.Size([404070, 1])


In [19]:
y_nonzero = pyg_data.y[pyg_data.y > 0].squeeze()

# Sort for percentile lookup
y_sorted, _ = torch.sort(y_nonzero)

n = y_sorted.numel()
percentile = lambda p: y_sorted[int(p * n)]

describe_dict = {
    'count': y_nonzero.numel(),
    'mean': y_nonzero.mean().item(),
    'std': y_nonzero.std(unbiased=False).item(),  # match pandas (ddof=0)
    'min': y_sorted[0].item(),
    '1%': percentile(0.01).item(),
    '2.5%': percentile(0.025).item(),
    '5%': percentile(0.05).item(),
    '25%': percentile(0.25).item(),
    '50%': y_nonzero.median().item(),
    '75%': percentile(0.75).item(),
    '95%': percentile(0.95).item(),
    '97.5%': percentile(0.975).item(),
    '99%': percentile(0.99).item(),
    'max': y_sorted[-1].item()
}

describe_aadt = pd.DataFrame(describe_dict, index=['AADT'])
describe_aadt

,count,mean,std,min,1%,2.5%,5%,25%,50%,75%,95%,97.5%,99%,max
AADT,110789,5853.708496,11087.919922,1.0,38.0,71.0,114.0,594.0,2329.0,6781.0,21901.0,31726.0,47984.0,196929.0


In [20]:
pcnt_value = describe_dict['1%']
num_below = (y_nonzero < pcnt_value).sum().item()
print(f"Number of nodes with AADT < 1th percentile ({pcnt_value:,.2f}): {num_below}")

Number of nodes with AADT < 1th percentile (38.00): 1096


In [21]:
p99_value = describe_dict['99%']
num_above_99 = (y_nonzero > p99_value).sum().item()
print(f"Number of nodes with AADT > 99th percentile ({p99_value:,.2f}): {num_above_99}")

Number of nodes with AADT > 99th percentile (47,984.00): 1107


In [22]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Convert to numpy
X_np = pyg_data.x.numpy()

# Fit and transform 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

# Replace in graph
pyg_data.x = torch.tensor(X_scaled, dtype=torch.float)
print(pyg_data.x.size())

torch.Size([404070, 18])


In [23]:
cat_cols  = ['Type_Operation','RURAL_URBAN','Terrain_Type','MedianType','AccessControl', ] #'CO_NAME'
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    pyg_data[col+'_encoded'] = le.fit_transform(pyg_data[col])
    encoders[col] = le  # Save for inverse transform later if needed

In [24]:
print(encoders)
encoders['Type_Operation']

{'Type_Operation': LabelEncoder(), 'RURAL_URBAN': LabelEncoder(), 'Terrain_Type': LabelEncoder(), 'MedianType': LabelEncoder(), 'AccessControl': LabelEncoder()}


LabelEncoder()

In [25]:
cat_dims = [int(len(encoders[col].classes_)) for col in cat_cols]
emb_dims = [min(50, (n + 1) // 2) for n in cat_dims]          # common embedding rule of thumb
print(f'cat_dims: {cat_dims}')
print(f'emb_dims_ideas 1: {emb_dims}')

cat_dims: [3, 2, 3, 8, 3]
emb_dims_ideas 1: [2, 1, 2, 4, 2]


In [26]:
encoded_categorical_cols = [
    'Type_Operation_encoded',
    'RURAL_URBAN_encoded',
    'Terrain_Type_encoded',
    'MedianType_encoded',
    'AccessControl_encoded',
    #'CO_NAME_encoded'
]

for col in encoded_categorical_cols:
    if isinstance(pyg_data[col], np.ndarray):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)
    elif isinstance(pyg_data[col], list):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)

In [27]:
from sklearn.model_selection import train_test_split

# nodes with measured AADT
has_aadt_mask = pyg_data.y.squeeze() > 0
y_nonzero = pyg_data.y.squeeze()[has_aadt_mask]

# 1st percentile cutoff
percentile_1 = torch.quantile(y_nonzero, 0.01) #1th percentile

# filter labeled_indices using 1st percentile threshold
labeled_indices = (pyg_data.y.squeeze() > percentile_1).nonzero(as_tuple=True)[0]

# Train/Val/Test split from labeled nodes only (no outliers)
train_idx, temp_idx = train_test_split(labeled_indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# boolean masks (all nodes) - initializing
num_nodes = pyg_data.num_nodes
pyg_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# train/validation/test masks
pyg_data.train_mask[train_idx] = True
pyg_data.val_mask[val_idx] = True
pyg_data.test_mask[test_idx] = True

## Modeling

In [28]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, summary
torch.manual_seed(42)

In [29]:
def mean_absolute_percentage_error_(y_true, y_pred):
    # Avoid division by zero
    epsilon = 1e-10
    return torch.mean(torch.abs((y_true - y_pred) / (y_true + epsilon))) * 100

In [30]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [31]:
def sinusoidal_functional_encoding(delta_f, edge_dim=16):
    """
    Encodes a single functional class difference value as a sinusoidal vector.

    Args:
        delta_f  : int, absolute F_SYSTEM difference between two nodes (0-6)
                   0 = same class (e.g. arterial <-> arterial)
                   6 = maximum mismatch (interstate=1 <-> local=7)
        edge_dim : int, output vector size (must be even)

    Returns:
        torch.Tensor of shape [edge_dim]
    """
    encoding = torch.zeros(edge_dim)
    for i in range(edge_dim // 2):
        # Frequency base of 1000 spreads the encoding well over a 0-6 range
        freq = 1000 ** (2 * i / edge_dim)
        encoding[2 * i]     = math.sin(delta_f / (freq + 1e-6))
        encoding[2 * i + 1] = math.cos(delta_f / (freq + 1e-6))
    return encoding

In [32]:
def attach_edge_attr(data, edge_dim=16):
    """
    Computes sinusoidal edge encodings from data.F_SYSTEM and attaches
    them to data.edge_attr.

    F_SYSTEM coding assumed:
        1 = Interstate
        2 = Freeway / Expressway
        3 = Principal Arterial
        4 = Minor Arterial
        5 = Major Collector
        6 = Minor Collector
        7 = Local

    Args:
        data     : PyG Data object with data.F_SYSTEM and data.edge_index
        edge_dim : int, sinusoidal encoding dimension (must be even)

    Returns:
        data object with data.edge_attr attached, shape [num_edges, edge_dim]
    """
    # Sanity check
    data.F_SYSTEM = torch.tensor(data.F_SYSTEM)
    unique_classes = data.F_SYSTEM.unique().tolist()
    print(f"F_SYSTEM unique values found: {sorted(unique_classes)}")
    print(f"Expected range: 1-7. Max possible delta_f: {int(data.F_SYSTEM.max() - data.F_SYSTEM.min())}")

    src, dst = data.edge_index          # each shape [num_edges]
    f_system  = data.F_SYSTEM.long().to(device)    # [num_nodes]

    src_class = f_system[src]           # F_SYSTEM of source node, per edge
    dst_class = f_system[dst]           # F_SYSTEM of dest node, per edge
    delta_f   = (src_class - dst_class).abs()  # shape [num_edges], values 0-6

    print(f"delta_f range: min={delta_f.min().item()}, max={delta_f.max().item()}")
    print(f"Computing sinusoidal encodings for {delta_f.shape[0]:,} edges...")

    # Build encoding matrix [num_edges, edge_dim]
    edge_attr = torch.stack([
        sinusoidal_functional_encoding(int(df.item()), edge_dim)
        for df in delta_f
    ])  # shape [num_edges, edge_dim]

    data.edge_attr = edge_attr
    print(f"data.edge_attr attached: shape {data.edge_attr.shape}")
    return data

In [33]:
import copy
def train_and_evaluate(data, model, epochs=2000, lr=1e-3, save_path='best_model_checkpoint.pt'):
    print(f"Training device: {device}")
    model = model.to(device)
    data = data.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_val_rmse = float('inf')
    best_model_state = None
    mape_at_best_val_rmse = float('inf')

    for epoch in range(epochs):

        # -----------------------------------------------------------------
        # Training step
        # -----------------------------------------------------------------
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # -----------------------------------------------------------------
        # Evaluation step
        # Fresh forward pass AFTER the optimizer step so metrics reflect
        # the updated weights, not the pre-update weights.
        # -----------------------------------------------------------------
        model.eval()
        with torch.no_grad():
            out = model(data)  # fresh pass with updated weights

            train_pred = out[data.train_mask]
            val_pred   = out[data.val_mask]
            test_pred  = out[data.test_mask]

            train_true = data.y[data.train_mask]
            val_true   = data.y[data.val_mask]
            test_true  = data.y[data.test_mask]

            train_loss = loss_fn(train_pred, train_true)
            val_loss   = loss_fn(val_pred,   val_true)
            test_loss  = loss_fn(test_pred,  test_true)

            # RMSE
            train_rmse = torch.sqrt(train_loss).item()
            val_rmse   = torch.sqrt(val_loss).item()
            test_rmse  = torch.sqrt(test_loss).item()

            # MAPE
            train_mape = mean_absolute_percentage_error_(train_true, train_pred).item()
            val_mape   = mean_absolute_percentage_error_(val_true,   val_pred).item()
            test_mape  = mean_absolute_percentage_error_(test_true,  test_pred).item()

        # -----------------------------------------------------------------
        # Checkpoint: save best model by validation RMSE
        # -----------------------------------------------------------------
        if val_rmse < best_val_rmse:
            best_val_rmse          = val_rmse
            mape_at_best_val_rmse  = val_mape
            best_model_state       = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, save_path)

        # -----------------------------------------------------------------
        # Progress logging every 50 epochs
        # -----------------------------------------------------------------
        if (epoch + 1) % 50 == 0:
            print(f"\nProgress @ Epoch {epoch + 1}:")
            print(f"  Best Val RMSE so far:              {best_val_rmse:.4f}")
            print(f"  MAPE @ Best Validation Checkpoint: {mape_at_best_val_rmse:.2f}%\n")
            print(f"  Train RMSE: {train_rmse:.4f}  |  Train MAPE: {train_mape:.2f}%")
            print(f"  Val RMSE:   {val_rmse:.4f}  |  Val MAPE:   {val_mape:.2f}%")
            print(f"  Test RMSE:  {test_rmse:.4f}  |  Test MAPE:  {test_mape:.2f}%\n")

    print(f"\nBest model saved with Val RMSE: {best_val_rmse:.4f} | "
          f"Val MAPE: {mape_at_best_val_rmse:.2f}% ")

    return best_val_rmse, mape_at_best_val_rmse

In [41]:
import copy

def train_and_evaluate_es(data, model, device, epochs=2000, lr=1e-3,
                          save_path='best_model_checkpoint.pt', patience=200, scheduler_patience=50):
    """
    Full-batch training with:
    - MAPE-based checkpointing (with RMSE guard)
    - Early stopping when val MAPE doesn't improve for `patience` epochs
    - LR scheduler that halves LR when val MAPE plateaus
    """

    print(f"Training device: {device}")

    data = data.to(device)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=scheduler_patience,
    )
    loss_fn = nn.MSELoss()

    # Tracking
    best_val_rmse = float('inf')
    best_val_mape = float('inf')
    rmse_at_best_mape = float('inf')
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):

        # -----------------------------------------------------------------
        # Training step (full batch)
        # -----------------------------------------------------------------
        model.train()
        optimizer.zero_grad()

        out = model(data).squeeze()

        loss = loss_fn(out[data.train_mask], data.y.squeeze()[data.train_mask])
        loss.backward()
        optimizer.step()

        # -----------------------------------------------------------------
        # Evaluation step
        # -----------------------------------------------------------------
        model.eval()
        with torch.no_grad():
            pred = model(data).squeeze()
            y = data.y.squeeze()

            train_pred, train_true = pred[data.train_mask], y[data.train_mask]
            val_pred,   val_true   = pred[data.val_mask],   y[data.val_mask]
            test_pred,  test_true  = pred[data.test_mask],  y[data.test_mask]

            train_rmse = torch.sqrt(loss_fn(train_pred, train_true)).item()
            val_rmse   = torch.sqrt(loss_fn(val_pred,   val_true)).item()
            test_rmse  = torch.sqrt(loss_fn(test_pred,  test_true)).item()

            train_mape = mean_absolute_percentage_error_(train_true, train_pred).item()
            val_mape   = mean_absolute_percentage_error_(val_true,   val_pred).item()
            test_mape  = mean_absolute_percentage_error_(test_true,  test_pred).item()

        # Step the scheduler based on val MAPE
        scheduler.step(val_mape)

        # -----------------------------------------------------------------
        # Track best RMSE (for the guard)
        # -----------------------------------------------------------------
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse

        # -----------------------------------------------------------------
        # Checkpoint on MAPE, guarded by RMSE
        # -----------------------------------------------------------------
        if val_mape < best_val_mape and val_rmse < best_val_rmse * 1.15:
            best_val_mape = val_mape
            rmse_at_best_mape = val_rmse
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, save_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        # -----------------------------------------------------------------
        # Early stopping
        # -----------------------------------------------------------------
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping at epoch {epoch} — no MAPE improvement for {patience} epochs")
            print(f"Best Val MAPE: {best_val_mape:.2f}% | RMSE: {rmse_at_best_mape:.4f}")
            break

        # -----------------------------------------------------------------
        # Progress logging
        # -----------------------------------------------------------------
        if epoch % 50 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"\nProgress @ Epoch {epoch} (LR: {current_lr:.1e}, no improvement: {epochs_without_improvement}):")
            print(f"  Best Val MAPE so far: {best_val_mape:.2f}%  (RMSE at that point: {rmse_at_best_mape:.4f})")
            print(f"  Best Val RMSE so far: {best_val_rmse:.4f}\n")
            print(f"  Train RMSE: {train_rmse:.4f}  |  Train MAPE: {train_mape:.2f}%")
            print(f"  Val RMSE:   {val_rmse:.4f}  |  Val MAPE:   {val_mape:.2f}%")
            print(f"  Test RMSE:  {test_rmse:.4f}  |  Test MAPE:  {test_mape:.2f}%\n")

    print(f"\nBest model saved — Val MAPE: {best_val_mape:.2f}% | RMSE: {rmse_at_best_mape:.4f}")

    return best_val_mape, rmse_at_best_mape

In [42]:
class GNNModel(nn.Module):
    def __init__(
        self,
        continuous_dims,   # number of continuous features in data.x
        gnn_in_dims,       # hidden dim for GNN layers
        gnn_out_dims,      # output dim of final GNN layer
        ln_in_dims,        # hidden dim of MLP regression head
        edge_dim=16,       # must match edge_dim used in attach_edge_attr
    ):
        super().__init__()

        # -----------------------------------------------------------------
        # Categorical embedding layers
        # -----------------------------------------------------------------
        self.embeddings = nn.ModuleDict({
            'Type_Operation_encoded': nn.Embedding(3, 3),
            'RURAL_URBAN_encoded':    nn.Embedding(2, 2),
            'Terrain_Type_encoded':   nn.Embedding(3, 3),
            'MedianType_encoded':     nn.Embedding(8, 5),
            'AccessControl_encoded':  nn.Embedding(3, 3),
        })

        total_embed_dim = sum(emb.embedding_dim for emb in self.embeddings.values())
        input_dim = total_embed_dim + continuous_dims

        # -----------------------------------------------------------------
        # Edge MLP: sinusoidal encoding vector -> scalar gate in (0, 1)
        # Sigmoid ensures output is always a valid soft weight.
        # Edges with delta_f=0 (same class) should learn gates near 1.
        # Edges with delta_f=6 (interstate <-> local) should learn gates near 0.
        # -----------------------------------------------------------------
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

        # -----------------------------------------------------------------
        # GNN layers
        # -----------------------------------------------------------------
        self.conv1 = GCNConv(input_dim,    gnn_in_dims)
        self.conv2 = GCNConv(gnn_in_dims,  gnn_in_dims)
        self.conv3 = GCNConv(gnn_in_dims,  gnn_out_dims)

        # -----------------------------------------------------------------
        # Residual skip connections
        # -----------------------------------------------------------------
        self.skip1 = nn.Linear(input_dim,   gnn_in_dims)
        self.skip2 = nn.Linear(gnn_in_dims, gnn_in_dims)
        self.skip3 = nn.Linear(gnn_in_dims, gnn_out_dims)

        # -----------------------------------------------------------------
        # MLP regression head
        # -----------------------------------------------------------------
        self.regressor = nn.Sequential(
            nn.Linear(gnn_out_dims, ln_in_dims),
            nn.ReLU(),
            nn.Linear(ln_in_dims, 1),
        )

    def forward(self, data):
        # -----------------------------------------------------------------
        # 1. Build node feature matrix from embeddings + continuous features
        # -----------------------------------------------------------------
        embedded = []
        for col in self.embeddings:
            x_cat = data[col].long()
            embedded.append(self.embeddings[col](x_cat))
        x_cat_embed = torch.cat(embedded, dim=1)

        x_cont = data.x
        x = torch.cat([x_cat_embed, x_cont], dim=1)  # [num_nodes, input_dim]

        # -----------------------------------------------------------------
        # 2. Compute scalar edge weights from sinusoidal edge encodings
        #    edge_attr : [num_edges, edge_dim]  (precomputed, fixed)
        #    edge_weight: [num_edges]            (learned, changes each step)
        # -----------------------------------------------------------------
        edge_weight = self.edge_mlp(data.edge_attr).squeeze(-1)  # [num_edges]

        # -----------------------------------------------------------------
        # 3. GNN message passing with learned functional-class edge weights
        # -----------------------------------------------------------------
        x1 = F.relu(self.conv1(x,  data.edge_index, edge_weight)) + self.skip1(x)
        x2 = F.relu(self.conv2(x1, data.edge_index, edge_weight)) + self.skip2(x1)
        x3 = F.relu(self.conv3(x2, data.edge_index, edge_weight)) + self.skip3(x2)

        return F.softplus(self.regressor(x3))

In [36]:
EDGE_DIM = 16
model_data = attach_edge_attr(pyg_data, edge_dim=EDGE_DIM)

F_SYSTEM unique values found: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
Expected range: 1-7. Max possible delta_f: 6
delta_f range: min=0, max=6
Computing sinusoidal encodings for 1,361,537 edges...
data.edge_attr attached: shape torch.Size([1361537, 16])


In [37]:
model_data['edge_attr']

tensor([[0., 1., 0.,  ..., 1., 0., 1.],
        [0., 1., 0.,  ..., 1., 0., 1.],
        [0., 1., 0.,  ..., 1., 0., 1.],
        ...,
        [0., 1., 0.,  ..., 1., 0., 1.],
        [0., 1., 0.,  ..., 1., 0., 1.],
        [0., 1., 0.,  ..., 1., 0., 1.]])

In [48]:
continuous_dims = model_data.x.size(1)
model = GNNModel(
    continuous_dims = continuous_dims,
    gnn_in_dims = 256,
    gnn_out_dims = 128,
    ln_in_dims = 64,
    edge_dim = EDGE_DIM,
)

In [49]:
save_path = 'checkpoints/'

In [50]:
#train_and_evaluate(model_data, model, epochs=25000, lr=1e-3,
#                   save_path=save_path + 'gcn_update_cp_nbc.pt')
#

In [54]:
train_and_evaluate_es(model_data, model, device, epochs=15000, lr= 6.3e-05,
                          save_path=save_path + 'gcn_update_cp_nbc_with_edge_data.pt', patience=4000, scheduler_patience=300)

Training device: cuda

Progress @ Epoch 50 (LR: 6.3e-05, no improvement: 49):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1631.7025  |  Train MAPE: 40.62%
  Val RMSE:   2383.0466  |  Val MAPE:   43.71%
  Test RMSE:  2322.4929  |  Test MAPE:  42.85%


Progress @ Epoch 100 (LR: 6.3e-05, no improvement: 99):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1629.8523  |  Train MAPE: 40.48%
  Val RMSE:   2385.3333  |  Val MAPE:   43.57%
  Test RMSE:  2324.2810  |  Test MAPE:  42.71%


Progress @ Epoch 150 (LR: 6.3e-05, no improvement: 149):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1629.4530  |  Train MAPE: 40.50%
  Val RMSE:   2385.1833  |  Val MAPE:   43.59%
  Test RMSE:  2324.3806  |  Test MAPE:  42.72%


Progress @ Epoch 200 (LR: 6.3e-05, no improvement: 199):
  Best Val MAPE so far: 39


Progress @ Epoch 1450 (LR: 3.9e-06, no improvement: 1449):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1622.5980  |  Train MAPE: 40.40%
  Val RMSE:   2384.0171  |  Val MAPE:   43.50%
  Test RMSE:  2324.1714  |  Test MAPE:  42.65%


Progress @ Epoch 1500 (LR: 3.9e-06, no improvement: 1499):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1622.4949  |  Train MAPE: 40.40%
  Val RMSE:   2384.0071  |  Val MAPE:   43.50%
  Test RMSE:  2324.1560  |  Test MAPE:  42.65%


Progress @ Epoch 1550 (LR: 2.0e-06, no improvement: 1549):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1622.4358  |  Train MAPE: 40.40%
  Val RMSE:   2384.0012  |  Val MAPE:   43.50%
  Test RMSE:  2324.1487  |  Test MAPE:  42.65%


Progress @ Epoch 1600 (LR: 2.0e-06, no improvement: 1599):
  Best Val MAPE so far: 39.99%  (RMSE


Progress @ Epoch 2850 (LR: 1.2e-07, no improvement: 2849):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1621.7457  |  Train MAPE: 40.39%
  Val RMSE:   2383.9033  |  Val MAPE:   43.50%
  Test RMSE:  2324.0640  |  Test MAPE:  42.64%


Progress @ Epoch 2900 (LR: 1.2e-07, no improvement: 2899):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1621.7396  |  Train MAPE: 40.39%
  Val RMSE:   2383.9019  |  Val MAPE:   43.50%
  Test RMSE:  2324.0635  |  Test MAPE:  42.64%


Progress @ Epoch 2950 (LR: 1.2e-07, no improvement: 2949):
  Best Val MAPE so far: 39.99%  (RMSE at that point: 2600.2087)
  Best Val RMSE so far: 2382.8743

  Train RMSE: 1621.7334  |  Train MAPE: 40.39%
  Val RMSE:   2383.9006  |  Val MAPE:   43.50%
  Test RMSE:  2324.0630  |  Test MAPE:  42.65%


Progress @ Epoch 3000 (LR: 1.2e-07, no improvement: 2999):
  Best Val MAPE so far: 39.99%  (RMSE

(39.992706298828125, 2600.208740234375)

## Model Evaluation

In [55]:
# Load best model
model.load_state_dict(torch.load(save_path + 'gcn_update_cp_nbc_with_edge_data.pt'))
model = model.to(device)
model.eval()

GNNModel(
  (embeddings): ModuleDict(
    (Type_Operation_encoded): Embedding(3, 3)
    (RURAL_URBAN_encoded): Embedding(2, 2)
    (Terrain_Type_encoded): Embedding(3, 3)
    (MedianType_encoded): Embedding(8, 5)
    (AccessControl_encoded): Embedding(3, 3)
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (conv1): GCNConv(34, 256)
  (conv2): GCNConv(256, 256)
  (conv3): GCNConv(256, 128)
  (skip1): Linear(in_features=34, out_features=256, bias=True)
  (skip2): Linear(in_features=256, out_features=256, bias=True)
  (skip3): Linear(in_features=256, out_features=128, bias=True)
  (regressor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [59]:
from sklearn.metrics import r2_score
loss_fn = nn.MSELoss()

def evaluate_split(data, model, device, mask):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        preds = model(data).squeeze()
        trues = data.y.squeeze()
    return preds[mask], trues[mask]

In [60]:
train_pred, train_true = evaluate_split(model_data, model, device, model_data.train_mask)
val_pred,   val_true   = evaluate_split(model_data, model, device, model_data.val_mask)
test_pred,  test_true  = evaluate_split(model_data, model, device, model_data.test_mask)

for name, pred, true in [('Train', train_pred, train_true),
                          ('Val',   val_pred,   val_true),
                          ('Test',  test_pred,  test_true)]:
    rmse = torch.sqrt(loss_fn(pred, true)).item()
    mape = mean_absolute_percentage_error_(true, pred).item()
    mae  = torch.mean(torch.abs(pred - true)).item()
    r2   = r2_score(true.cpu().numpy(), pred.cpu().numpy())
    print(f"{name:5s} — RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}% | R²: {r2:.4f}")

Train — RMSE: 1889.0271 | MAE: 998.7245 | MAPE: 37.48% | R²: 0.9710
Val   — RMSE: 2600.2087 | MAE: 1139.7788 | MAPE: 39.99% | R²: 0.9461
Test  — RMSE: 2532.7083 | MAE: 1160.5657 | MAPE: 39.03% | R²: 0.9487


In [63]:
def get_predictions(data, model, device, mask):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        out = model(data).squeeze()
        trues = data.y.squeeze()
    return out[mask].cpu().numpy(), trues[mask].cpu().numpy()

train_pred, train_true = get_predictions(model_data, model, device, model_data.train_mask)
val_pred,   val_true   = get_predictions(model_data, model, device, model_data.val_mask)
test_pred,  test_true  = get_predictions(model_data, model, device, model_data.test_mask)

# --- Define bins ---
bin_edges = [
    0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000,
    5500, 6000, 6500, 7000, 7500, 8000, 8500, 9000, 9500, 10000,
    20000, 35000, 55000, 85000, 125000, np.inf
]
bin_labels = [
    "< 500", "500 - 1000", "1000 - 1500", "1500 - 2000", "2000 - 2500", "2500 - 3000",
    "3000 - 3500", "3500 - 4000", "4000 - 4500", "4500 - 5000", "5000 - 5500", "5500 - 6000",
    "6000 - 6500", "6500 - 7000", "7000 - 7500", "7500 - 8000", "8000 - 8500", "8500 - 9000",
    "9000 - 9500", "9500 - 10000", "10000 - 20000",
    "20000 - 35000", "35000 - 55000", "55000 - 85000", "85000 - 125000", ">125000"
]

# --- Metrics function ---
def bin_metrics(group):
    t = group["y_true"].values
    p = group["y_pred"].values
    n = len(t)
    rmse = np.sqrt(np.mean((p - t) ** 2))
    mae  = np.mean(np.abs(p - t))
    mape = np.mean(np.abs((t - p) / np.clip(t, 1, None))) * 100
    r2   = r2_score(t, p) if n > 1 else np.nan
    return pd.Series({"N": n, "RMSE": rmse, "MAE": mae, "MAPE": mape,})

# --- Run for each split + combined ---
splits = {
    "Train": (train_pred, train_true),
    "Val":   (val_pred,   val_true),
    "Test":  (test_pred,  test_true),
    "All":   (np.concatenate([train_pred, val_pred, test_pred]),
              np.concatenate([train_true, val_true, test_true])),
}

for split_name, (pred, true) in splits.items():
    mask = true > 0
    df = pd.DataFrame({"y_true": true[mask], "y_pred": pred[mask]})
    df["aadt_bin"] = pd.cut(df["y_true"], bins=bin_edges, labels=bin_labels,
                            right=False, include_lowest=True)
    results = df.groupby("aadt_bin", observed=False).apply(bin_metrics)
    overall = bin_metrics(df)
    overall.name = "OVERALL"
    results = pd.concat([results, overall.to_frame().T])
    print(f"\n{'='*80}")
    print(f"  {split_name} Set Metrics by AADT Bin")
    print(f"{'='*80}")
    print(results.to_string())


  Train Set Metrics by AADT Bin
                      N          RMSE          MAE       MAPE
< 500           16397.0    260.740143   156.371460  84.066236
500 - 1000       9083.0    458.811310   293.713745  41.052279
1000 - 1500      5757.0    598.216431   411.081512  33.476695
1500 - 2000      4453.0    685.754883   510.195648  29.356071
2000 - 2500      3494.0    826.265137   605.529968  27.132756
2500 - 3000      3143.0    903.482727   665.102234  24.260275
3000 - 3500      2991.0   1063.607788   772.501343  23.942983
3500 - 4000      2302.0   1122.967773   870.093262  23.255901
4000 - 4500      2176.0   1289.874512   950.174927  22.515383
4500 - 5000      2149.0   1335.128662  1000.199036  21.023926
5000 - 5500      1792.0   1344.709106  1012.795837  19.329241
5500 - 6000      1510.0   1427.949219  1088.730103  18.977948
6000 - 6500      1452.0   1547.197144  1173.640503  18.763687
6500 - 7000      1245.0   1459.070679  1119.059326  16.568381
7000 - 7500      1347.0   1744.651123

In [ ]:
# Before training (once) ---
#
#   EDGE_DIM = 16
#   data = attach_edge_attr(data, edge_dim=EDGE_DIM)
#   # data.edge_attr is now shape [1361537, 16]
#
# --- Instantiate model ---
#
#   model = GNNModel(
#       continuous_dims=19,   # matches data.x shape[1]
#       gnn_in_dims=64,
#       gnn_out_dims=32,
#       ln_in_dims=16,
#       edge_dim=EDGE_DIM,    # must match attach_edge_attr
#   )
#
# --- Training loop (standard, nothing special needed) ---
#
#   optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#   criterion = nn.MSELoss()
#
#   for epoch in range(num_epochs):
#       model.train()
#       optimizer.zero_grad()
#       pred = model(data)
#       loss = criterion(pred[train_mask], data.y[train_mask])
#       loss.backward()
#       optimizer.step()
#
# --- After training: inspect what the model learned ---
#
#   model.inspect_edge_weights(edge_dim=EDGE_DIM)